In [1]:
import pandas as pd
import numpy as np
import sklearn.preprocessing
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from scipy.stats import zscore
import matplotlib.pyplot as plt
import seaborn as sns
import phate
from sklearn.neighbors import KNeighborsRegressor
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter1d
from scipy.ndimage import gaussian_filter
import os

import rmm
import cupy
import cudf
import cupy as cp
from rmm.allocators.cupy import rmm_cupy_allocator
import anndata as an
import scanpy as sc
import rapids_singlecell as rsc
import scvi

from cuml.manifold import UMAP
from cuml.decomposition import TruncatedSVD

# Enable `managed_memory`
rmm.reinitialize(
    managed_memory=True,
    pool_allocator=False,
)
cp.cuda.set_allocator(rmm_cupy_allocator)

# Load Trajectories (.csv)

In [2]:
%%time
fpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/cell_cycle/outputs/gene_trajectories_100.csv"
df = pd.read_csv(fpath)
print(f"{df.shape=}")
df.head()

df.shape=(114900, 15909)
CPU times: user 6min 31s, sys: 35.1 s, total: 7min 6s
Wall time: 7min 36s


,Unnamed: 0,time,A1BG,A1BG-AS1,A2M,A4GALT,A4GNT,AAAS,AACS,AADAT,...,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,cell_idx
0,0,0,49.516611,5.610672,13.397070,5.698189,3.981307,16.784324,11.826024,7.300885,...,55.565021,4.492255,2.927349,10.780328,5.642261,40.657299,153.982223,8.175810,34.069625,0
1,1,1,49.526262,5.618392,13.394447,5.684149,3.979548,16.777699,11.817051,7.326938,...,55.569033,4.498559,2.913986,10.814142,5.577114,40.622668,153.469253,8.153397,34.077446,0
2,2,2,49.543578,5.626243,13.392648,5.670422,3.978114,16.770564,11.809276,7.352886,...,55.567403,4.504593,2.900912,10.847847,5.513321,40.590913,152.967239,8.131673,34.086679,0
3,3,3,49.568043,5.634157,13.391444,5.656916,3.976959,16.762898,11.802588,7.378738,...,55.560464,4.510322,2.888090,10.881349,5.450783,40.561777,152.475409,8.110502,34.097097,0
4,4,4,49.599868,5.642130,13.391023,5.643654,3.976124,16.754664,11.796973,7.404435,...,55.547552,4.515748,2.875547,10.914615,5.389579,40.535240,151.993507,8.089947,34.108600,0


# Make Adata

In [5]:
%%time

outdir = os.path.dirname(fpath)
basename = os.path.basename(fpath).replace(".csv", "")
outpath = f"{outdir}/{basename}.h5ad"
print(f"Saving to: {outpath}")

# make obs
obs = df[['time', 'cell_idx']].copy()
obs['cell_by_t'] = obs['cell_idx'].astype(str) + ":" + obs['time'].astype(str)
obs = obs.set_index('cell_by_t')

# make X
X = df.drop(columns=['Unnamed: 0', 'time', 'cell_idx'])

# make adata
adata = an.AnnData(
    X=X, obs=obs,
    var=pd.DataFrame(index=X.columns),
)

# save to disk
adata.write(outpath)
adata

Saving to: /nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/cell_cycle/outputs/gene_trajectories_100.h5ad
CPU times: user 30.9 s, sys: 19.8 s, total: 50.7 s
Wall time: 1min 21s


AnnData object with n_obs × n_vars = 114900 × 15906
    obs: 'time', 'cell_idx'

In [4]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [ ]:
# # trajectories
# fpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/cell_cycle/outputs/pca_trajectories_100.npy"
# trajectories = np.load(fpath)
# print(f"{trajectories.shape=}")

# # whitened data
# fpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/cell_cycle/outputs/whitened_data.npy"
# data = np.load(fpath)
# print(f"{data.shape=}")